# Thành viên 4 — Phân tích tương quan MovieLens

Ngưỡng tối thiểu: **100 lượt rating mỗi phim**. Tải `movie_summary.csv` lên Colab trước khi chạy ô đọc dữ liệu.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

# Nếu movie_summary.csv chưa có trong Colab, chạy ô này và chọn file CSV.
uploaded = files.upload()


In [ ]:
# Đọc dữ liệu tóm tắt phim
from pathlib import Path
path = Path('movie_summary.csv')
if not path.exists():
    csv_files = list(Path('.').glob('*.csv'))
    if len(csv_files) == 1:
        path = csv_files[0]
    else:
        raise FileNotFoundError('Hãy tải movie_summary.csv lên Colab rồi chạy lại ô này.')

df = pd.read_csv(path)
required = {'movieId', 'rating_count', 'average_rating'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Thiếu cột: {missing}. Hãy chọn đúng file movie_summary.csv.')
df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce')
df['average_rating'] = pd.to_numeric(df['average_rating'], errors='coerce')
df = df.dropna(subset=['rating_count', 'average_rating'])
df = df[df['rating_count'] >= 1].copy()
filtered = df[df['rating_count'] >= 20].copy()
print('Số phim hợp lệ:', len(df))
print('Số phim có ít nhất 100 lượt rating:', len(filtered))
print('Tỷ lệ giữ lại: {:.2f}%'.format(len(filtered)/len(df)*100))

In [ ]:
# Tương quan toàn bộ phim
pearson_all = df['rating_count'].corr(df['average_rating'], method='pearson')
spearman_all = df['rating_count'].corr(df['average_rating'], method='spearman')
plt.figure(figsize=(9,5))
plt.scatter(df['rating_count'], df['average_rating'], alpha=.4, s=16)
plt.xlabel('Số lượt đánh giá (rating_count)')
plt.ylabel('Điểm đánh giá trung bình (average_rating)')
plt.title(f'Toàn bộ phim | Pearson r={pearson_all:.4f}')
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()
print('Pearson toàn bộ =', round(pearson_all, 4))
print('Spearman toàn bộ =', round(spearman_all, 4))

In [ ]:
# Tương quan nhóm phim đạt ngưỡng 100 lượt rating
pearson_filtered = filtered['rating_count'].corr(filtered['average_rating'], method='pearson')
spearman_filtered = filtered['rating_count'].corr(filtered['average_rating'], method='spearman')
plt.figure(figsize=(9,5))
plt.scatter(filtered['rating_count'], filtered['average_rating'], alpha=.55, s=24)
plt.xlabel('Số lượt đánh giá (rating_count)')
plt.ylabel('Điểm đánh giá trung bình (average_rating)')
plt.title(f'Phim có ít nhất 100 lượt rating (n={len(filtered)}) | Pearson r={pearson_filtered:.4f}')
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()
print('Pearson sau lọc =', round(pearson_filtered, 4))
print('Spearman sau lọc =', round(spearman_filtered, 4))

In [ ]:
# Bảng so sánh và lưu kết quả
comparison = pd.DataFrame([
    {'Tập dữ liệu':'Toàn bộ phim','Điều kiện':'rating_count >= 1','Số phim':len(df),'Tỷ lệ giữ lại (%)':100.0,'Pearson':pearson_all,'Spearman':spearman_all},
    {'Tập dữ liệu':'Sau lọc','Điều kiện':'rating_count >= 100','Số phim':len(filtered),'Tỷ lệ giữ lại (%)':len(filtered)/len(df)*100,'Pearson':pearson_filtered,'Spearman':spearman_filtered}
])
display(comparison.round(4))
print('Thay đổi Pearson: {:+.4f}'.format(pearson_filtered-pearson_all))
comparison.to_csv('bang_so_sanh_tuong_quan.csv', index=False, encoding='utf-8-sig')
print('Đã lưu bang_so_sanh_tuong_quan.csv')